# BioSkills Lab — AI/ML Capstone
## Did the Foundation Model Actually Help?

**Course:** AI for Genomics in the Foundation Model Era  
**Task:** Cell type annotation — PCA vs scVI vs Geneformer  

[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

---

### What you will do
Take the same set of single cells and annotate their types using three increasingly complex methods:
1. **PCA + kNN** — classical, fast, interpretable baseline
2. **scVI + kNN** — deep generative model, handles batch effects
3. **Geneformer + kNN** — foundation model pretrained on 30M cells *(GPU only, optional)*

At the end you compare all three on the same metric and answer: **was the complexity worth it?**

### Runtime
- Steps 1–4 (PCA + scVI): CPU or GPU both fine
- Step 5 (Geneformer): requires **GPU** — Runtime > Change runtime type > T4 GPU

### Step 0: Install dependencies
We need `scanpy` for single-cell processing and `scvi-tools` for the deep generative model. This takes about 2 minutes.

In [ ]:
!pip install -q scanpy scvi-tools anndata scikit-learn umap-learn

In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import torch
import time
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

print(f'scanpy: {sc.__version__}')
print(f'scvi-tools: {scvi.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

### Step 1: Load dataset

We try to download the **scIB PBMC benchmark** (~150MB) from figshare — a multi-donor PBMC dataset used in the landmark Luecken et al. 2022 benchmarking paper. It has known cell type labels and multiple donors (batches), making it ideal for comparing batch-aware vs batch-naive methods.

If the download fails, we fall back to the **PBMC 3k** dataset built into scanpy (~6MB, single donor).

**What to expect:** The dataset should have 5–14 cell types (T cells, B cells, NK cells, monocytes, dendritic cells etc.) across thousands of cells.

In [ ]:
import urllib.request

USE_SCIB = False
try:
    print('Downloading scIB PBMC benchmark (~150MB)...')
    urllib.request.urlretrieve('https://figshare.com/ndownloader/files/24539842', 'pbmc_scib.h5ad')
    adata = sc.read_h5ad('pbmc_scib.h5ad')
    if 'cell_type' not in adata.obs.columns:
        raise ValueError('No cell_type column')
    USE_SCIB = True
    print(f'scIB loaded: {adata.shape[0]} cells, {adata.shape[1]} genes')
except Exception as e:
    print(f'scIB unavailable ({e})')
    print('Using PBMC 3k built-in dataset...')
    adata = sc.datasets.pbmc3k()
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
    adata = adata[adata.obs.pct_counts_mt < 5].copy()
    # Assign rough cell type labels from marker-based annotation
    # (will be replaced by leiden clusters as proxy labels)
    adata.obs['cell_type'] = 'unknown'
    print(f'PBMC 3k loaded: {adata.shape[0]} cells, {adata.shape[1]} genes')

print('\nCell type counts:')
print(adata.obs['cell_type'].value_counts())

### Step 2: Preprocessing

We do three things here:
1. **Store raw counts** — scVI requires unnormalised integer counts, so we save them before any transformation
2. **Select highly variable genes** — we keep the 2000 most variable genes to reduce noise and computation. We use `flavor='seurat'` (not `seurat_v3`) which is more robust across different input types
3. **Normalise and log-transform** — required for PCA. Each cell is scaled to 10,000 total counts, then log-transformed

We then fix the same train/test split for all three methods — this is critical for a fair comparison.

In [ ]:
# Store raw counts before any normalisation
if 'counts' not in adata.layers:
    adata.layers['counts'] = adata.X.copy()

# For PBMC 3k fallback: generate cell type labels via quick clustering
if (adata.obs['cell_type'] == 'unknown').all():
    print('Generating cell type labels via clustering...')
    adata_tmp = adata.copy()
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)
    sc.pp.highly_variable_genes(adata_tmp, n_top_genes=2000, subset=True, flavor='seurat')
    sc.pp.scale(adata_tmp)
    sc.tl.pca(adata_tmp)
    sc.pp.neighbors(adata_tmp)
    sc.tl.leiden(adata_tmp, resolution=0.5)
    adata.obs['cell_type'] = adata_tmp.obs['leiden'].values
    print(f'Generated {adata.obs["cell_type"].nunique()} clusters as proxy labels')

# Select highly variable genes — use 'seurat' flavor (robust for all input types)
batch_key = 'batch' if 'batch' in adata.obs.columns else None
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False,
                              flavor='seurat', batch_key=batch_key)

# Normalise and log transform for PCA
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

# Encode cell type labels
le = LabelEncoder()
y = le.fit_transform(adata.obs['cell_type'])

# Fix the train/test split — same split used for ALL methods
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)
y_train, y_test = y[train_idx], y[test_idx]
print(f'Train: {len(train_idx)} cells, Test: {len(test_idx)} cells')
print(f'Cell types: {list(le.classes_)}')

### Step 3: Method 1 — PCA + kNN

PCA compresses 2000 gene dimensions into 50 principal components that capture most of the variance. A k-nearest neighbour classifier then labels each test cell based on its 15 closest neighbours in PCA space.

**This is your baseline.** Every more complex method must beat this to justify its additional cost.

**What to expect:** ~85–95% accuracy on well-separated cell types. Runs in seconds.

In [ ]:
t0 = time.time()
sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
X_pca = adata.obsm['X_pca']
pca_time = time.time() - t0

knn = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn.fit(X_pca[train_idx], y_train)
y_pred_pca = knn.predict(X_pca[test_idx])

acc_pca = accuracy_score(y_test, y_pred_pca)
f1_pca  = f1_score(y_test, y_pred_pca, average='macro')
print(f'PCA + kNN | Accuracy: {acc_pca:.3f} | Macro-F1: {f1_pca:.3f} | Time: {pca_time:.1f}s')

### Step 4: Method 2 — scVI + kNN

scVI is a variational autoencoder trained specifically on raw count data. It learns a 20-dimensional latent representation of each cell while modelling batch effects as a nuisance variable — meaning cells from different donors are projected into the same latent space regardless of technical differences.

**Key difference from PCA:** scVI uses the full probabilistic count model (negative binomial), handles batch effects explicitly, and captures non-linear relationships.

**What to expect:** Training takes 2–5 minutes. Performance should be similar to or better than PCA, especially if the dataset has multiple batches/donors.

In [ ]:
t0 = time.time()
adata_scvi = adata.copy()
adata_scvi.X = adata_scvi.layers['counts']

setup_kwargs = {'layer': 'counts'}
if batch_key:
    setup_kwargs['batch_key'] = batch_key
scvi.model.SCVI.setup_anndata(adata_scvi, **setup_kwargs)
model_scvi = scvi.model.SCVI(adata_scvi, n_latent=20, n_layers=2)
model_scvi.train(max_epochs=200, early_stopping=True,
                  early_stopping_patience=20, batch_size=256)

X_scvi = model_scvi.get_latent_representation()
adata.obsm['X_scVI'] = X_scvi
scvi_time = time.time() - t0

knn_scvi = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn_scvi.fit(X_scvi[train_idx], y_train)
y_pred_scvi = knn_scvi.predict(X_scvi[test_idx])

acc_scvi = accuracy_score(y_test, y_pred_scvi)
f1_scvi  = f1_score(y_test, y_pred_scvi, average='macro')
print(f'scVI + kNN | Accuracy: {acc_scvi:.3f} | Macro-F1: {f1_scvi:.3f} | Time: {scvi_time:.1f}s')

### Step 5: Method 3 — Geneformer (GPU required, optional)

Geneformer is a transformer pretrained on 30 million single-cell transcriptomes. Each cell is represented as a rank-ordered sequence of genes (highest expressed first), and the model has learned general gene network relationships from this massive pretraining.

We extract the cell embeddings (512 dimensions) from the pretrained model without any fine-tuning — this is **zero-shot transfer**.

**What to expect:** Slower than scVI (requires tokenisation + forward pass through a transformer). On standard PBMC tasks, performance is often similar to scVI. The advantage of Geneformer shows most clearly on rare cell types and cross-tissue generalisation.

> If you are on CPU this cell will skip automatically.

In [ ]:
GENEFORMER_AVAILABLE = False
acc_gf, f1_gf, gf_time = 0.0, 0.0, 0.0

if not torch.cuda.is_available():
    print('No GPU — skipping Geneformer. Switch to T4 GPU runtime to run this step.')
else:
    try:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'geneformer'], check=True, capture_output=True)
        from geneformer import TranscriptomeTokenizer, EmbExtractor
        import os, scipy.sparse as sp

        t0 = time.time()
        os.makedirs('gf_input', exist_ok=True)
        os.makedirs('gf_tokens', exist_ok=True)
        os.makedirs('gf_embs', exist_ok=True)

        adata_gf = adata.copy()
        adata_gf.X = adata_gf.layers['counts']
        if sp.issparse(adata_gf.X): adata_gf.X = adata_gf.X.toarray()
        adata_gf.X = adata_gf.X.astype(int)
        adata_gf.write_loom('gf_input/pbmc.loom')

        tk = TranscriptomeTokenizer({'cell_type': 'cell_type'}, nproc=2)
        tk.tokenize_data('gf_input/', 'gf_tokens/', 'pbmc', file_format='loom')

        embex = EmbExtractor(model_type='Pretrained', num_classes=0,
                              emb_mode='cell', cell_emb_style='mean_pool',
                              emb_layer=-1, emb_label=['cell_type'],
                              forward_batch_size=100, nproc=2)
        embs = embex.extract_embs('ctheodoris/Geneformer',
                                   'gf_tokens/pbmc.dataset', 'gf_embs/', 'pbmc')
        X_gf = embs.values
        adata.obsm['X_geneformer'] = X_gf
        gf_time = time.time() - t0

        knn_gf = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
        knn_gf.fit(X_gf[train_idx], y_train)
        y_pred_gf = knn_gf.predict(X_gf[test_idx])

        acc_gf = accuracy_score(y_test, y_pred_gf)
        f1_gf  = f1_score(y_test, y_pred_gf, average='macro')
        GENEFORMER_AVAILABLE = True
        print(f'Geneformer + kNN | Accuracy: {acc_gf:.3f} | Macro-F1: {f1_gf:.3f} | Time: {gf_time:.1f}s')
    except Exception as e:
        print(f'Geneformer failed: {e}\nContinuing with PCA and scVI results only.')

### Step 6: Final comparison

We compare all methods on:
- **Macro-F1** — the primary metric. Averages F1 across all cell types equally, so rare cell types matter as much as common ones. More informative than accuracy on imbalanced data.
- **Runtime** — computational cost matters. A 1% F1 improvement that takes 100x longer is rarely worth it in practice.

**How to interpret:** If scVI Macro-F1 ≈ PCA Macro-F1, the additional complexity of a deep generative model was not necessary for this task. If Geneformer ≈ scVI, the foundation model added nothing beyond what a simpler batch-correction method could do.

In [ ]:
rows = [
    {'Method': 'PCA + kNN',  'Accuracy': acc_pca,  'Macro-F1': f1_pca,  'Time (s)': pca_time},
    {'Method': 'scVI + kNN', 'Accuracy': acc_scvi, 'Macro-F1': f1_scvi, 'Time (s)': scvi_time},
]
if GENEFORMER_AVAILABLE:
    rows.append({'Method': 'Geneformer + kNN', 'Accuracy': acc_gf, 'Macro-F1': f1_gf, 'Time (s)': gf_time})

results = pd.DataFrame(rows).sort_values('Macro-F1', ascending=False)
print('\n' + '='*55)
print('FINAL RESULTS: Did the Foundation Model Actually Help?')
print('='*55)
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#38bdf8', '#4ade80', '#a78bfa'][:len(results)]
axes[0].bar(results['Method'], results['Macro-F1'], color=colors)
axes[0].set_ylabel('Macro-F1'); axes[0].set_title('Classification Performance')
axes[0].set_ylim(0, 1); axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(results['Method'], results['Time (s)'], color=colors)
axes[1].set_ylabel('Runtime (seconds)'); axes[1].set_title('Computational Cost')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

### Step 7: UMAP visualisation

UMAPs let you visually inspect the quality of each representation. A good embedding should:
- Show **tight, well-separated clusters** for distinct cell types (top row)
- Show **no visible separation by donor/batch** (bottom row) — if batches form separate islands, the method failed to correct for technical variation

scVI should show better batch mixing than PCA if multiple donors are present.

In [ ]:
methods_to_plot = [('PCA', 'X_pca'), ('scVI', 'X_scVI')]
if GENEFORMER_AVAILABLE:
    methods_to_plot.append(('Geneformer', 'X_geneformer'))

n_rows = 2 if batch_key else 1
fig, axes = plt.subplots(n_rows, len(methods_to_plot), figsize=(6*len(methods_to_plot), 5*n_rows))
if n_rows == 1: axes = axes.reshape(1, -1)

for col, (name, key) in enumerate(methods_to_plot):
    if key not in adata.obsm: continue
    sc.pp.neighbors(adata, use_rep=key, n_neighbors=15, key_added=f'nn_{name}')
    sc.tl.umap(adata, neighbors_key=f'nn_{name}', key_added=f'umap_{name}')
    sc.pl.embedding(adata, basis=f'umap_{name}', color='cell_type',
                    ax=axes[0, col], show=False, title=f'{name} — Cell Type')
    if batch_key:
        sc.pl.embedding(adata, basis=f'umap_{name}', color=batch_key,
                        ax=axes[1, col], show=False, title=f'{name} — Donor')

plt.tight_layout(); plt.show()

### Your conclusions

Fill in your results below:

```
RESULTS:
  PCA Macro-F1:         ___
  scVI Macro-F1:        ___
  Geneformer Macro-F1:  ___ (if run)

INTERPRETATION:
  Did scVI outperform PCA? By how much?
  Was the gain worth the extra training time?
  Did Geneformer add anything beyond scVI?
  Which method would you use in a real single-cell project?
  What surprised you most?
```

---
**BioSkills Lab** — [bioskillslab.dev](https://bioskillslab.dev)